In [ ]:
import os
import re
import json
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm

from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score, accuracy_score
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

sns.set_theme(style="darkgrid", palette="muted")
plt.rcParams.update({"figure.dpi": 120, "font.family": "DejaVu Sans"})
ACCENT  = "#7C3AED"
ACCENT2 = "#06B6D4"
RED     = "#EF4444"
GREEN   = "#22C55E"
AMBER   = "#F59E0B"
SIGNER_COLORS = {
    "Signer_A_numeric":    ACCENT,
    "Signer_B_dash":       RED,
    "Signer_C_underscore": ACCENT2,
    "Signer_D_bisindo":    AMBER,
}

In [ ]:
# ============================================================
# 0 -- Config
# ============================================================
# Phase 0 diagnostics. NO TRAINING happens in this notebook -- it only
# inspects the feature data to explain the 58 pp gap between the stratified
# 5-fold result (99.16%) and the LOSO result (41.33%), and to decide what
# the single Phase 1 training run should contain.
DATA_DIR        = "features"
SEQUENCE_LENGTH = 30
FEATURES_DIM    = 447
MAX_SAMPLES     = 110
RANDOM_STATE    = 42

POSE_SLICE = slice(0, 99)     # 33 landmarks x 3
FACE_SLICE = slice(99, 321)   # 74 landmarks x 3
LH_SLICE   = slice(321, 384)  # 21 landmarks x 3
RH_SLICE   = slice(384, 447)  # 21 landmarks x 3
BLOCKS = {"pose": POSE_SLICE, "face": FACE_SLICE, "left_hand": LH_SLICE, "right_hand": RH_SLICE}

# A pair closer than this (relative L2, see Section 1) is treated as a
# near-duplicate candidate. Deliberately loose: the printed distance
# histogram is the real evidence, this is only for counting.
NEAR_DUP_REL_THRESHOLD = 0.05

verdict = {}


def classify_signer(fname: str) -> str:
    stem = os.path.splitext(fname)[0]
    stem = re.sub(r"\s*\(\d+\)$", "", stem)
    if stem.upper().startswith("BISINDO_"):
        return "Signer_D_bisindo"
    if re.fullmatch(r"\d+", stem):
        return "Signer_A_numeric"
    if re.search(r"-\d+$", stem):
        return "Signer_B_dash"
    if re.search(r"_\d+$", stem):
        return "Signer_C_underscore"
    return "Signer_UNK"

In [ ]:
# ============================================================
# 1 -- Load the balanced dataset (same rule as the training notebooks)
# ============================================================
# Balanced, not raw: the balanced set is what actually trains and tests, so
# it is the set whose leakage and domain shift matter.
print("Loading dataset with signer-group labels...")
actions = sorted([d for d in os.listdir(DATA_DIR) if os.path.isdir(os.path.join(DATA_DIR, d))])
num_classes = len(actions)
label_map = {a: i for i, a in enumerate(actions)}

sequences, labels, groups, fnames = [], [], [], []
for action in actions:
    action_path = os.path.join(DATA_DIR, action)
    for f in tqdm([f for f in os.listdir(action_path) if f.endswith(".npy")],
                  desc=f"  {action:<15}", leave=False):
        seq = np.load(os.path.join(action_path, f))
        if seq.shape == (SEQUENCE_LENGTH, FEATURES_DIM):
            sequences.append(seq)
            labels.append(label_map[action])
            groups.append(classify_signer(f))
            fnames.append(f"{action}/{f}")

X = np.array(sequences, dtype=np.float32)
y_int = np.array(labels)
group_all = np.array(groups)
fname_all = np.array(fnames)

idx = []
for i in range(num_classes):
    idx.extend(np.where(y_int == i)[0][:MAX_SAMPLES])
X, y_int, group_all, fname_all = X[idx], y_int[idx], group_all[idx], fname_all[idx]
print(f"  Balanced set: {X.shape[0]} sequences, {num_classes} classes")
print("  Per-signer counts:", {g: int((group_all == g).sum()) for g in sorted(set(group_all.tolist()))})

In [ ]:
# ============================================================
# 2 -- CHECK 0.1: exact and near-duplicate detection
# ============================================================
# Two distinct risks:
#   within-signer duplicates  -> inflate the 99.32% stratified-split result
#   cross-signer duplicates   -> inflate LOSO, meaning true LOSO is worse
# Exact duplicates are found by hashing raw bytes; near-duplicates by an
# exact all-pairs L2 distance (one matmul, ~4400^2, cheap at this size).
print("\n" + "=" * 60)
print("CHECK 0.1 -- DUPLICATE DETECTION")
print("=" * 60)

X_flat = X.reshape(len(X), -1)

hashes = {}
exact_dupes = []
for i, row in enumerate(X_flat):
    h = hash(row.tobytes())
    if h in hashes:
        exact_dupes.append((hashes[h], i))
    else:
        hashes[h] = i
print(f"Exact duplicate pairs: {len(exact_dupes)}")
for a, b in exact_dupes[:10]:
    tag = "SAME-signer" if group_all[a] == group_all[b] else "CROSS-signer"
    print(f"  [{tag}] {fname_all[a]}  ==  {fname_all[b]}")

# All-pairs relative L2:  ||a-b|| / mean(||a||, ||b||)
norms = np.linalg.norm(X_flat, axis=1)
gram = X_flat @ X_flat.T
sq = norms[:, None] ** 2 + norms[None, :] ** 2 - 2 * gram
np.maximum(sq, 0, out=sq)
dist = np.sqrt(sq, out=sq)
denom = (norms[:, None] + norms[None, :]) / 2.0
rel = dist / np.maximum(denom, 1e-8)
np.fill_diagonal(rel, np.inf)

nn_rel = rel.min(axis=1)
print(f"\nNearest-neighbour relative distance over {len(X)} sequences:")
for p in [0, 1, 5, 25, 50]:
    print(f"  {p:>2}th percentile: {np.percentile(nn_rel, p):.4f}")

iu = np.triu_indices(len(X), k=1)
near_mask = rel[iu] < NEAR_DUP_REL_THRESHOLD
n_near = int(near_mask.sum())
near_i, near_j = iu[0][near_mask], iu[1][near_mask]
n_near_cross = int((group_all[near_i] != group_all[near_j]).sum())
print(f"\nPairs below rel-distance {NEAR_DUP_REL_THRESHOLD}: {n_near} "
      f"({n_near_cross} cross-signer, {n_near - n_near_cross} same-signer)")
for a, b in list(zip(near_i, near_j))[:10]:
    tag = "SAME-signer" if group_all[a] == group_all[b] else "CROSS-signer"
    print(f"  [{tag}] rel={rel[a, b]:.4f}  {fname_all[a]}  ~  {fname_all[b]}")

verdict["exact_duplicate_pairs"] = len(exact_dupes)
verdict["near_duplicate_pairs"] = n_near
verdict["near_duplicate_pairs_cross_signer"] = n_near_cross

fig, ax = plt.subplots(figsize=(9, 4))
ax.hist(nn_rel, bins=80, color=ACCENT, edgecolor="white", linewidth=0.4)
ax.axvline(NEAR_DUP_REL_THRESHOLD, color=RED, lw=1.5, linestyle="--",
           label=f"near-dup threshold {NEAR_DUP_REL_THRESHOLD}")
ax.set_xlabel("Nearest-neighbour relative L2 distance")
ax.set_ylabel("Sequences")
ax.set_title("Nearest-Neighbour Distance Distribution (duplicates would pile up near 0)",
             fontweight="bold")
ax.legend()
plt.tight_layout()
plt.savefig("diag_nn_distance_hist.png", dpi=120)
plt.show()

del gram, sq, dist, denom, rel

In [ ]:
# ============================================================
# 3 -- CHECK 0.2: is signer identity separable in the features?
# ============================================================
# Sequences are mean/std-pooled over the 30 frames (447*2 = 894 features):
# pooling keeps the static body-geometry information that carries signer
# identity, which is exactly what this check is about.
print("\n" + "=" * 60)
print("CHECK 0.2 -- SIGNER SEPARABILITY / DOMAIN SHIFT")
print("=" * 60)

pooled = np.concatenate([X.mean(axis=1), X.std(axis=1)], axis=1)
pooled_z = StandardScaler().fit_transform(pooled)
pca = PCA(n_components=50, random_state=RANDOM_STATE)
emb = pca.fit_transform(pooled_z)
print(f"PCA(50) explains {pca.explained_variance_ratio_.sum()*100:.1f}% of pooled-feature variance")

# Silhouette: how well-separated are the clusters under each labelling?
# signer >> class means the geometry encodes WHO is signing more strongly
# than WHAT is signed.
sil_signer = silhouette_score(emb, group_all, random_state=RANDOM_STATE)
sil_class = silhouette_score(emb, y_int, random_state=RANDOM_STATE)
print(f"\nSilhouette by SIGNER: {sil_signer:+.4f}")
print(f"Silhouette by CLASS : {sil_class:+.4f}")

# Can a trivial linear model recover signer identity from the landmarks?
Xtr, Xte, gtr, gte = train_test_split(
    emb, group_all, test_size=0.25, stratify=group_all, random_state=RANDOM_STATE)
clf = LogisticRegression(max_iter=2000, random_state=RANDOM_STATE)
clf.fit(Xtr, gtr)
signer_acc = accuracy_score(gte, clf.predict(Xte))
majority = max((group_all == g).mean() for g in set(group_all.tolist()))
print(f"\nLinear signer classifier accuracy: {signer_acc*100:.2f}%  "
      f"(majority-class baseline {majority*100:.2f}%)")

verdict["silhouette_signer"] = float(sil_signer)
verdict["silhouette_class"] = float(sil_class)
verdict["linear_signer_classifier_acc"] = float(signer_acc)
verdict["signer_majority_baseline"] = float(majority)

fig, ax = plt.subplots(figsize=(7, 6))
for g in sorted(set(group_all.tolist())):
    m = group_all == g
    ax.scatter(emb[m, 0], emb[m, 1], s=6, alpha=0.45,
               color=SIGNER_COLORS.get(g, "gray"), label=g.replace("Signer_", ""))
ax.set_xlabel("PC1")
ax.set_ylabel("PC2")
ax.set_title("Pooled Landmark Features, Coloured by Signer", fontweight="bold")
ax.legend(markerscale=2)
plt.tight_layout()
plt.savefig("diag_pca_by_signer.png", dpi=120)
plt.show()

In [ ]:
# ============================================================
# 4 -- CHECK 0.3: per-signer landmark statistics
# ============================================================
# Signer_B scored 19.33% in LOSO against 46-53% for the others. If B differs
# in MediaPipe detection rate or landmark spread, that is a recording-setup
# outlier rather than a "hard signer", which changes how it gets written up.
print("\n" + "=" * 60)
print("CHECK 0.3 -- PER-SIGNER LANDMARK STATISTICS")
print("=" * 60)

signers = sorted(set(group_all.tolist()))
print(f"\n{'Signer':<22}{'n':>6}" + "".join(f"{b + ' miss%':>14}" for b in BLOCKS))
miss_table = {}
for g in signers:
    Xg = X[group_all == g]
    row = {}
    line = f"{g:<22}{len(Xg):>6}"
    for bname, sl in BLOCKS.items():
        rate = float(np.all(Xg[:, :, sl] == 0, axis=-1).mean())
        row[bname] = rate
        line += f"{rate*100:>13.2f}%"
    miss_table[g] = row
    print(line)

print(f"\n{'Signer':<22}" + "".join(f"{b + ' |mag|':>14}" for b in BLOCKS))
mag_table = {}
for g in signers:
    Xg = X[group_all == g]
    row = {}
    line = f"{g:<22}"
    for bname, sl in BLOCKS.items():
        blk = Xg[:, :, sl]
        present = blk[~np.all(blk == 0, axis=-1)]
        mag = float(np.linalg.norm(present, axis=-1).mean()) if len(present) else 0.0
        row[bname] = mag
        line += f"{mag:>14.3f}"
    mag_table[g] = row
    print(line)

verdict["per_signer_missing_rate"] = miss_table
verdict["per_signer_block_magnitude"] = mag_table

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
w = 0.2
xp = np.arange(len(BLOCKS))
for k, g in enumerate(signers):
    axes[0].bar(xp + (k - 1.5) * w, [miss_table[g][b] * 100 for b in BLOCKS], w,
                label=g.replace("Signer_", ""), color=SIGNER_COLORS.get(g, "gray"))
    axes[1].bar(xp + (k - 1.5) * w, [mag_table[g][b] for b in BLOCKS], w,
                label=g.replace("Signer_", ""), color=SIGNER_COLORS.get(g, "gray"))
for ax, ttl, ylb in [(axes[0], "Missing-Landmark Rate by Signer", "% of frames with block absent"),
                     (axes[1], "Mean Landmark Magnitude by Signer", "mean |xyz| (shoulder units)")]:
    ax.set_xticks(xp)
    ax.set_xticklabels(list(BLOCKS))
    ax.set_title(ttl, fontweight="bold")
    ax.set_ylabel(ylb)
    ax.legend(fontsize=8)
plt.tight_layout()
plt.savefig("diag_per_signer_stats.png", dpi=120)
plt.show()

In [ ]:
# ============================================================
# 5 -- VERDICT: what Phase 1 should contain
# ============================================================
# Turns the three checks into the concrete decision they were run to make,
# so the Phase 1 run is chosen by evidence rather than by guess.
print("\n" + "=" * 60)
print("PHASE 0 VERDICT")
print("=" * 60)

leak = verdict["exact_duplicate_pairs"] > 0 or verdict["near_duplicate_pairs"] > 0
if leak:
    print(f"[!] LEAKAGE: {verdict['exact_duplicate_pairs']} exact + "
          f"{verdict['near_duplicate_pairs']} near-duplicate pairs "
          f"({verdict['near_duplicate_pairs_cross_signer']} cross-signer).")
    print("    -> De-duplicate before Phase 1; the 99.32% is inflated to some degree.")
else:
    print("[ok] No exact or near-duplicate pairs. The 99.32% is not duplicate-driven,")
    print("     and the LOSO 41.33% is not softened by cross-signer copies.")

shift = verdict["linear_signer_classifier_acc"] > 0.90 or sil_signer > sil_class
if shift:
    print(f"\n[!] DOMAIN SHIFT: signer is linearly recoverable at "
          f"{verdict['linear_signer_classifier_acc']*100:.1f}% "
          f"(baseline {verdict['signer_majority_baseline']*100:.1f}%), "
          f"silhouette signer {sil_signer:+.3f} vs class {sil_class:+.3f}.")
    print("    -> Signer identity is strongly encoded in the landmarks themselves.")
    print("    -> ADD the delta-feature arm to Phase 1 (frame-to-frame differences,")
    print("       which drop static body geometry and keep motion): 16 runs, not 12.")
else:
    print(f"\n[ok] Signer identity is not trivially recoverable "
          f"({verdict['linear_signer_classifier_acc']*100:.1f}%).")
    print("     -> SKIP the delta-feature arm. Phase 1 stays at 12 runs.")

b_miss = np.mean([miss_table["Signer_B_dash"][b] for b in BLOCKS])
others_miss = np.mean([miss_table[g][b] for g in signers if g != "Signer_B_dash" for b in BLOCKS])
if b_miss > others_miss * 1.5:
    print(f"\n[!] Signer_B missing-landmark rate {b_miss*100:.1f}% vs "
          f"{others_miss*100:.1f}% for the others.")
    print("    -> B's 19.33% LOSO fold is a tracking/recording-setup outlier.")
    print("       Report it as such, not as an intrinsically hard signer.")
else:
    print(f"\n[ok] Signer_B missing-landmark rate {b_miss*100:.1f}% is comparable to "
          f"{others_miss*100:.1f}% for the others.")
    print("     -> B's low LOSO fold is NOT explained by tracking failure.")

with open("signer_diagnostics.json", "w") as fh:
    json.dump(verdict, fh, indent=2)
print("\nSaved -> signer_diagnostics.json")